<a href="https://colab.research.google.com/github/ja2faded/ds2002-fa26/blob/main/2026_10_16_%E2%80%94_Walmart_Warm_Up_SKU_Mess_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
from google.colab import drive
drive.mount ('/content/drive')
PATH = '/content/drive/MyDrive/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
print(cat.head())
print(cat.shape)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
             sku          product_name            category subcategory  \
0          PT-12  pop-tarts strawberry     Breakfast Foods         NaN   
1     SKU#459812   Strawberry Pop-Tart           Breakfast   Pop-Tarts   
2  POPTART-STRAW  Pop-Tarts Strawberry           Breakfast         NaN   
3  PT_STRAWBERRY  pop-tarts strawberry           breakfast   Pop-Tarts   
4         WTR-24    BOTTLED WATER 24PK  Emergency Supplies     Bottled   

   unit_cost           supplier package_size  
0       1.93  Johnson & Johnson       4-pack  
1       2.26           Roku Inc          4pk  
2       2.11           Sara Lee       6 pack  
3        NaN         Campbell's       4 pack  
4       2.64           Sara Lee       1 pack  
(45, 7)


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
pop = cat[cat['product_name'].str.lower().str.contains('pop')]
print('Pop-Tart rows: ', len(pop))
pop

Pop-Tart rows:  4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
pop[['sku', 'product_name', 'package_size']]

,sku,product_name,package_size
0,PT-12,pop-tarts strawberry,4-pack
1,SKU#459812,Strawberry Pop-Tart,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,4 pack


Pop-Tart rows that are a different product: None. All 4 Pop-Tart rows are strawberry, so there is no reason to separate flavor.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
canonical = {
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW'
}

n_before = len(cat)
cat['canonical_sku'] = cat['sku'].replace(canonical)
print("Before: ", cat['sku'].nunique())
print("After: ", cat['canonical_sku'].nunique())

Before:  45
After:  42


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print(cat['category'].value_counts(), '\n')
cat['category'] = cat['category'].str.strip().str.title()
print(cat['category'].value_counts())


category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64 

category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast Foods        1
Alco

Still 2 labels for same category: Breakfast and Breakfast Foods, Emergency and Emergency Supplies, Grocery and Groceries, Alcohol and Alcoholic Beverages, Baby and Baby Products

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
cat['normalized'] = (cat['product_name'].str.lower()
                         .str.replace(r'[^a-z ]', '', regex=True)
                         .str.split().apply(lambda w: ' '.join(sorted(w))))

candidates = (cat.groupby('normalized')['sku']
              .agg(list).loc[lambda s: s.str.len() > 1])
candidates

,sku
normalized,
aa batteries pack,"[BAT-AA4, BATT_AA_4PK]"
beer light pack,"[BEER-24, LTBEER24]"
board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
bread white,"[BREAD-WH, WHITEBREAD]"
butter peanut,"[PB-16, PNUTBTR-16]"
candles emergency,"[CNDL-EM, CANDLE-EMRG]"
canned soup,"[SOUP-CAN, CNSOUP-01]"
cards playing,"[CARDS-01, PLAYCARD-STD]"
chips pack variety,"[CHIPS-VP, CHIPVARIETY]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:

assert cat['canonical_sku'].notna().all()
assert len(cat) == n_before

### Write-up

a. Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

1. Bottled water is fragmented (3 SKUs: WTR-24, WATER24PK, WTR_24PK). Since they are all 24 packs, I would group them together (same product).

2. Board game monopoly  is also fragmented (3 SKUs: BG-MONO, MONOPOLY-01, GAME-MONO). Again, all are Monopoly , so I would group them together.


I would group them by repeating the canonical mapping in Q3 to give each one canonical SKU.


b. Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.


Consolidation is data cleaning when the SKUs differ in formatting or spelling, but refer to the same product; for example PT-12 and PT_STRAWBERRY.

It becomes a business decision when merging changes what counts as a product.

An example where I am unsure is about grouping POPTART-STRAW (listed as a 6 pack) with other strawberry poptart SKUs that are listed as a 4 pack, since they are the same flavor, but technically they are not the same exact product, as one has more poptarts inside. I would look into asking the distributor of the product if they consider different sized packages of the same products as one product or separate.



_your answer here_